In [46]:
import qutip as qt
import matplotlib.pyplot as plt
import numpy as np

from IPython.display import display

Define number of atoms in the ensemble and the Fock-state cutoffs for the Stokes photon number (ns) and the anti-Stokes photon number (nas).

In [47]:
atom_number = 3
ns_cutoff = 2
nas_cutoff = 2

In [48]:
# calculate the generalized rates/shifts for N atoms

def get_generalized_effective_parameters(atom_number, Omega_p, Omega_c, Delta_p, Delta_c, g_s, g_as, kappa_s, kappa_as, n_ph=0):
    # list of all m from 0 to N -> N+1 states
    m_vals = np.arange(atom_number + 1)
    
    # stark shift
    delta_F = Omega_p**2 / (4 * Delta_p) * (atom_number - m_vals) + g_s**2 / Delta_p * m_vals * (n_ph + 1)
    
    delta_e = g_s**2 / Delta_p
    delta_g = Omega_p**2 / (4 * Delta_p)
    
    # list of all m values for Raman coupling transitions -> m from 0 to N-1 (at state N the Rate vanishes)
    m_trans = np.arange(atom_number)
    
    lambda_eff = Omega_p * g_s / (2 * Delta_p) * np.sqrt((atom_number - m_trans) * (m_trans + 1) * (n_ph + 1))
    
    Gamma_s = 4 * lambda_eff**2 / kappa_s
    
    A_square = (Gamma_s + kappa_as)**2 + 4 * (Delta_c - delta_g)**2
    
    def X(m):
        return (Gamma_s + kappa_as)**2 - 4 * (Delta_c - delta_g) * (delta_e - delta_g) + (m + 1) * Omega_c**2
    
    def Y(m):
        return 2 * (Gamma_s + kappa_as) * (Delta_c - 2 * delta_g + delta_e)
    
    Gamma_as_nom = (m_trans + 1) * (atom_number - m_trans) * Omega_c**2 * g_as**2 * (Gamma_s[m_trans] + kappa_as) * (A_square + (m_trans + 1) * Omega_c**2)
    Gamma_as_denom = ((Delta_c - delta_e) * X(m_trans) + 4 * (Delta_c - delta_g) * (atom_number - m_trans) * g_as**2)**2 \
                    + ((Delta_c - delta_e) * Y(m_trans) + 2 * (Gamma_s[m_trans] + kappa_as) * (atom_number - m_trans) * g_as**2)**2
    
    Gamma_as = Gamma_as_nom / Gamma_as_denom
    
    return m_vals, delta_F, lambda_eff, Gamma_s, Gamma_as

In [50]:
# define the basis for a single atoms as fock basis -> Schwinger bosons
states = [qt.basis(4, i) for i in range(4)]

# atom basis states
state1 = states[0]
state2 = states[1]
state3 = states[2]
state4 = states[3]

# fock basis states for Stokes mode
basis_s = qt.fock(ns_cutoff, 0)
basis_as = qt.fock(nas_cutoff, 0)

# identity operators
I_atom = qt.qeye(4)
I_s = qt.qeye(ns_cutoff)
I_as = qt.qeye(nas_cutoff)
I = qt.tensor([I_atom] * atom_number + [I_s, I_as])

# function to build operators in total collective atomic Hilbert space
def atom_operator(op, atom, N):
    operators = []
    
    for i in range(N):
        if i == atom:
            operators.append(op)
        else:
            operators.append(I_atom)
            
    operators.append(I_s)
    operators.append(I_as)
    
    return qt.tensor(operators)

In [51]:
# single atom transition operators

sigma_11_local = qt.projection(4, 0, 0)
sigma_22_local = qt.projection(4, 1, 1)
sigma_33_local = qt.projection(4, 2, 2)
sigma_44_local = qt.projection(4, 3, 3)

sigma_12_local = qt.projection(4, 0, 1)
sigma_21_local = sigma_12_local.dag() 

sigma_13_local = qt.projection(4, 0, 2)
sigma_31_local = sigma_13_local.dag()

sigma_23_local = qt.projection(4, 1, 2)
sigma_32_local = sigma_23_local.dag()

sigma_14_local = qt.projection(4, 0, 3)
sigma_41_local = sigma_14_local.dag()

sigma_24_local = qt.projection(4, 1, 3)
sigma_42_local = sigma_24_local.dag()

sigma_list = [[sigma_11_local, sigma_12_local, sigma_13_local,sigma_14_local], [sigma_21_local, sigma_22_local, sigma_23_local, sigma_24_local], 
              [sigma_31_local, sigma_32_local, sigma_33_local, None], [sigma_41_local, sigma_42_local, None, sigma_44_local]]

# save sigmas as dictonary -> index directly describes operator subscripts, list would need unnecessary RAM
sigma = {}

sigma[(1, 1)] = sigma_11_local
sigma[(1, 2)] = sigma_12_local
sigma[(1, 3)] = sigma_13_local
sigma[(1, 4)] = sigma_14_local
sigma[(2, 1)] = sigma_21_local
sigma[(2, 2)] = sigma_22_local
sigma[(2, 3)] = sigma_23_local
sigma[(2, 4)] = sigma_24_local
sigma[(4, 1)] = sigma_41_local
sigma[(4, 2)] = sigma_42_local
sigma[(4, 4)] = sigma_44_local


In [52]:
# collective operators

sigma_11 = []
sigma_22 = []
sigma_33 = []
sigma_44 = []

sigma_12 = []
sigma_21 = []

sigma_13 = []
sigma_31 = []

sigma_23 = []
sigma_32 = []

sigma_14 = []
sigma_41 = []

sigma_24 = []
sigma_42 = []


# define operators in the N-particle atomic Hilbert space
for i in range(atom_number):
    sigma_11.append(atom_operator(sigma_list[0][0], i, atom_number))
    sigma_22.append(atom_operator(sigma_list[1][1], i, atom_number))
    sigma_33.append(atom_operator(sigma_list[2][2], i, atom_number))
    sigma_44.append(atom_operator(sigma_list[3][3], i, atom_number))
    
    sigma_12.append(atom_operator(sigma_list[0][1], i, atom_number))
    sigma_21.append(atom_operator(sigma_list[1][0], i, atom_number))
    
    sigma_13.append(atom_operator(sigma_list[0][2], i, atom_number))
    sigma_31.append(atom_operator(sigma_list[2][0], i, atom_number))
    
    sigma_23.append(atom_operator(sigma_list[1][2], i, atom_number))
    sigma_32.append(atom_operator(sigma_list[2][1], i, atom_number))
    
    sigma_14.append(atom_operator(sigma_list[0][3], i, atom_number))
    sigma_41.append(atom_operator(sigma_list[3][0], i, atom_number))
    
    sigma_24.append(atom_operator(sigma_list[1][3], i, atom_number))
    sigma_42.append(atom_operator(sigma_list[3][1], i, atom_number))

# define the collective operators
Sigma_11 = sum(sigma_11)
Sigma_22 = sum(sigma_22)
Sigma_33 = sum(sigma_33)
Sigma_44 = sum(sigma_44)

Sigma_12 = sum(sigma_12)
Sigma_21 = sum(sigma_21)

Sigma_13 = sum(sigma_13)
Sigma_31 = sum(sigma_31)

Sigma_23 = sum(sigma_23)
Sigma_32 = sum(sigma_32)

Sigma_14 = sum(sigma_14)
Sigma_41 = sum(sigma_41)

Sigma_24 = sum(sigma_24)
Sigma_42 = sum(sigma_42)

Sigma_list = [[Sigma_11, Sigma_12, Sigma_13, Sigma_14], [Sigma_21, Sigma_22, Sigma_23, Sigma_24], [Sigma_31, Sigma_32, Sigma_33, None], [Sigma_41, Sigma_42, None, Sigma_44]]

Sigma = {}

Sigma[(1, 1)] = Sigma_11
Sigma[(1, 2)] = Sigma_12
Sigma[(1, 3)] = Sigma_13
Sigma[(1, 4)] = Sigma_14
Sigma[(2, 1)] = Sigma_21
Sigma[(2, 2)] = Sigma_22
Sigma[(2, 3)] = Sigma_23
Sigma[(2, 4)] = Sigma_24
Sigma[(3, 1)] = Sigma_13
Sigma[(3, 2)] = Sigma_32
Sigma[(3, 3)] = Sigma_33
Sigma[(3, 4)] = None
Sigma[(4, 1)] = Sigma_41
Sigma[(4, 2)] = Sigma_42
Sigma[(4, 3)] = None
Sigma[(4, 4)] = Sigma_44

In [53]:
# define the bosonic operators in the total Hilbert space

a_s = qt.tensor([I_atom for _ in range(atom_number)] + [qt.destroy(ns_cutoff), I_as])
a_s_dag = a_s.dag()

a_as = qt.tensor([I_atom for _ in range(atom_number)] + [I_s, qt.destroy(nas_cutoff)])
a_as_dag = a_as.dag()

Ns = a_s_dag * a_s
Nas = a_as_dag * a_as


In [54]:
# total state information and initial state

print("State definition: |Atom_1, ..., Atom_N, Stokesn ati-Stokes>")
print(f"Full Hilbert-space dimensions: {I.shape}")
print(f"Total Hilbert-space dimension: {I.shape[0]}")
# initial state
psi_atoms = [qt.basis(4, 0) for _ in range(atom_number)]
psi_s     = qt.fock(ns_cutoff, 0)
psi_as    = qt.fock(nas_cutoff, 0)

psi0 = qt.tensor(psi_atoms + [psi_s, psi_as])

State definition: |Atom_1, ..., Atom_N, Stokesn ati-Stokes>
Full Hilbert-space dimensions: (256, 256)
Total Hilbert-space dimension: 256


In [55]:
# Hamiltonian

def get_Hamiltonian(Delta_p, Delta_c, Omega_p, Omega_c, g_s, g_as):
    
    H_atom = Delta_p * Sigma_44 + Delta_c * Sigma_33
    H_drive = (Omega_p/2) * (Sigma_41 + Sigma_14) + (Omega_c/2) * (Sigma_32 + Sigma_23)
    H_cav = g_s * (a_s_dag * Sigma_24 + a_s * Sigma_42) + g_as * (a_as_dag * Sigma_13 + a_as * Sigma_31)
    
    return H_atom + H_drive + H_cav

# def get_Hamiltonian(Delta_p, Omega_p, g_s, Sigma_dict):
    
#     H_atom = Delta_p * Sigma[(4, 4)]
#     H_int = (Omega_p/2) * (Sigma[(4, 1)] + Sigma[(1, 4)]) + g_s * (a_s_dag * Sigma[(2, 4)] + a_s * Sigma[(4, 2)])
    
#     return H_atom + H_int

In [56]:
# collapse operators for master equation

def collapse_operator_list(kappa_s, kappa_as, Gamma_41=None, Gamma_42=None, Gamma_31=None, Gamma_32=None, Gamma_d=None):

    c_ops = []

    c_ops.append(np.sqrt(kappa_s) * a_s)
    c_ops.append(np.sqrt(kappa_as) * a_as)
    
    c_ops.append(np.sqrt(Gamma_41) * Sigma_41) if Gamma_41 is not None else None
    c_ops.append(np.sqrt(Gamma_42) * Sigma_42) if Gamma_42 is not None else None
    c_ops.append(np.sqrt(Gamma_31) * Sigma_31) if Gamma_31 is not None else None
    c_ops.append(np.sqrt(Gamma_32) * Sigma_32) if Gamma_32 is not None else None
    c_ops.append(np.sqrt(Gamma_d) * (Sigma_11 - Sigma_22)/2) if Gamma_d is not None else None
    
    return c_ops
    

In [57]:
# solve the master equation

def solve_atomic_rate_model(Omega_p, Omega_c, Delta_p, Delta_c, g_s, g_as, kappa_s, kappa_as, Gamma_41=None, Gamma_42=None, Gamma_31=None, Gamma_32=None, Gamma_d=None, solver=None, t_list=None, ntraj=1500):
    
    # expectation values
    e_ops = [Sigma_44, Sigma_33, Sigma_22, Sigma_11, Ns, Nas]
    
    # Hamiltonian and collapse operators
    H = get_Hamiltonian(Delta_p,  Delta_c, Omega_p, Omega_c, g_s, g_as)
    c_ops = collapse_operator_list(kappa_s, kappa_as, Gamma_41, Gamma_42, Gamma_31, Gamma_32, Gamma_d)
    
    if t_list is None:
        t_list = np.linspace(0, 1500, 300)
    
    # if no solve is explicitely mentioned, use MCWF method, else exact solver
    if solver is None:
        data = qt.mcsolve(H, psi0, t_list, c_ops=c_ops, e_ops=e_ops, ntraj=ntraj)
    else:
        data = qt.mesolve(H, psi0, t_list, c_ops=c_ops, e_ops=e_ops)
        
    return t_list, data

In [58]:
# solve the model in the Dicke basis

def solve_dicke_rate_model(atom_number, Omega_p, Omega_c, Delta_p, Delta_c, g_s, g_as, kappa_s, kappa_as, t_list, n_ph=0):
    
    # |F_m> -> |F_{m+1}> at rate Gamma_s^m
    
    N = atom_number
    dim = N + 1
    
    # effective Raman couplings: m -> m+1
    m_trans = np.arange(N)
    
    # get generalized rates and Stark-shifts
    
    m_vals, delta_F, lambda_eff, Gamma_s, Gamma_as = get_generalized_effective_parameters(atom_number, Omega_p, Omega_c, Delta_p, Delta_c, g_s, g_as, kappa_s, kappa_as, n_ph)
    
    # initial state: all atoms in 1
    psi0 = qt.basis(dim, 0)
    rho0 = qt.ket2dm(psi0)
    
    # Dicke state number operator ->  M|m> = m|m>
    M = qt.Qobj(np.diag(m_vals))
    projectors = [qt.basis(dim, m) * qt.basis(dim, m).dag() for m in m_vals]
    
    # collapse from m -> m+1
    c_ops = []
    
    # Stokes: for each transition m -> m+1 define the collapse operator
    for m in range(N):
        ket_next = qt.basis(dim, m+1)
        bra_current = qt.basis(dim, m).dag()
        
        L_s = np.sqrt(Gamma_s[m]) * ket_next * bra_current
        c_ops.append(L_s)

    # anti-Stokes: for each transition m+1 -> m define the collapse operator
    for m in range(N):
        ket_next = qt.basis(dim, m)
        bra_current = qt.basis(dim, m+1).dag()
        
        L_as = np.sqrt(Gamma_as[m]) * ket_next * bra_current
        c_ops.append(L_as)
        
    H_eff = 0 * qt.qeye(dim)
    
    result = qt.mesolve(H_eff, rho0, t_list, c_ops=c_ops, e_ops=[M]+projectors)
    
    mean_m = result.expect[0]
    populations = np.array(result.expect[1:])
    
    return t_list, m_vals, Gamma_s, Gamma_as, mean_m, populations 

In [59]:
# numerics with data and plotting

def plot_solution(Omega_p, Omega_c, Delta_p, Delta_c, g_s, g_as, kappa_s, kappa_as, Gamma_41=None, Gamma_42=None, Gamma_31=None, Gamma_32=None, Gamma_d=None, solver="exact"):

    # get the data from the full system description
    t_list, data = solve_atomic_rate_model(Omega_p=Omega_p, Omega_c=Omega_c, Delta_p=Delta_p, Delta_c=Delta_c, g_s=g_s, g_as=g_as,
                                           kappa_s=kappa_s, kappa_as=kappa_as, Gamma_41=Gamma_41, Gamma_42=Gamma_42, Gamma_31=Gamma_31, Gamma_32=Gamma_32, Gamma_d=Gamma_d, solver=solver)

    pop4_expect = data.expect[0]
    pop3_expect = data.expect[1]
    pop2_expect = data.expect[2]
    pop1_expect = data.expect[3]
    populations = [pop1_expect, pop2_expect, pop3_expect, pop4_expect]
    Ns_expect = data.expect[4]
    Nas_expect = data.expect[5]

    # get the data from the Dicke basis
    
    t_list, m_vals, Gamma_s, Gamma_as, mean_m, dicke_populations = solve_dicke_rate_model(atom_number=atom_number, Omega_p=Omega_p, Omega_c=Omega_c, Delta_p=Delta_p, Delta_c=Delta_c,
                                                                                          g_s=g_s, g_as=g_as, kappa_s=kappa_s, kappa_as=kappa_as, t_list=t_list, n_ph=0)
    
    ns_eff = np.zeros_like(t_list)
    nas_eff = np.zeros_like(t_list)
    for m in range(atom_number):
        ns_eff += (Gamma_s[m] / kappa_s) * dicke_populations[m]
        nas_eff += (Gamma_as[m] / kappa_as) * dicke_populations[m+1]
    
    # plot the populations
    
    plt.figure(figsize=(6, 4))

    # requires Latex distribution installes on system
    try:
        plt.rcParams['text.usetex'] = True
    except Exception as e:
        plt.rcParams['text.usetex'] = True
        print(f"Latex distribution konnte nicht gefunden werden: {e}")

    for i, population in enumerate(populations, start=1):
        plt.plot(t_list, population, label=fr'$\langle S_{{{i}{i}}}\rangle$')
        
    plt.title("Population")
    plt.xlabel(r'$\Gamma t$')
    plt.ylabel(r'$\langle S_{ii}\rangle$')
    plt.legend(loc='upper right')
    plt.grid(True)
    plt.savefig("C:/Users/lauri/OneDrive/Desktop/Uni/Physik/Master/Thesis/Code/Results/Stokes_full_system_4_atoms.pdf")

    # populations of each Dicke state
    
    plt.figure(figsize=(6,4))
    
    for m, P_m in zip(m_vals, dicke_populations):
        plt.plot(t_list, P_m, label=fr"$\langle |{m}\rangle\langle {m}| \rangle$")
    
    plt.title(r"population of Dicke state $|m\rangle$")
    plt.xlabel(fr"$\Gamma t$")
    plt.ylabel(r"$\langle |m\rangle\langle m| \rangle$")
    plt.legend()
    plt.grid(True)
    plt.savefig("C:/Users/lauri/OneDrive/Desktop/Uni/Physik/Master/Thesis/Code/Results/Stokes_dicke_system_4_atoms.pdf")

    # compare the populations in 2
    
    plt.figure(figsize=(6, 4))
    
    plt.plot(t_list, pop2_expect, label='full model')
    plt.plot(t_list, mean_m, '--', label='Dicke model')    
    
    plt.title(r"mean population in state $|2\rangle$")
    plt.xlabel(r"$\Gamma t$")
    plt.ylabel("population")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("C:/Users/lauri/OneDrive/Desktop/Uni/Physik/Master/Thesis/Code/Results/Stokes_population_state_2_4_atoms.pdf")

    # plot the photon numbers
    
    plt.figure(figsize=(6, 4))
    plt.plot(t_list, Ns_expect, color="#005AA9", linestyle='-', label="Stokes (full system model)")
    plt.plot(t_list, Nas_expect, color="#ff7f0e", linestyle='-', label="anti-Stokes (full system model)")
    plt.plot(t_list, ns_eff, color="#027428", linestyle='--', label="Stokes (Dicke model)")
    plt.plot(t_list, nas_eff, color="#d1261a", linestyle='--', label="anti-Stokes (Dicke model)")
    plt.title("Stokes and Anti-Stokes Photon Numbers")
    plt.xlabel(r'$\Gamma t$')
    plt.ylabel(r'$\langle N \rangle$')
    plt.legend(loc="upper right")
    plt.grid(True)
    plt.savefig("C:/Users/lauri/OneDrive/Desktop/Uni/Physik/Master/Thesis/Code/Results/both_photon_numbers_4_atoms.pdf")

    # plot the effective rates over all possible Dicke states |m>
    
    plt.figure(figsize=(6,4))
    
    plt.plot(m_vals[:-1], Gamma_s, 'o-', label=r"theory $\Gamma_{{s}}^{(m)}$")
    plt.plot(m_vals[:-1], Gamma_as, 'o-', label=r"theory $\Gamma_{{as}}^{(m)}$")
    plt.title(r"effective $\Gamma$ for each Dicke state $|m\rangle$")
    plt.xlabel(r"Dicke state number $m$")
    plt.ylabel(r"effective Raman-rate")
    plt.grid(True)
    plt.legend()
    plt.savefig("C:/Users/lauri/OneDrive/Desktop/Uni/Physik/Master/Thesis/Code/Results/Stokes_effective_Raman_rate_4_atoms.pdf")


    plt.show()


In [ ]:
Gamma = 1
Omega_p = 3 * Gamma
Omega_c = 0.05 * Gamma
g_s = 1 * Gamma
g_as = 1 * Gamma
Delta_p = 20 * Gamma
Delta_c = g_s**2 / Delta_p
kappa_s = 1000 * Gamma
kappa_as = 1000 * Gamma

plot_solution(Omega_p=Omega_p, Omega_c=Omega_c, Delta_p=Delta_p, Delta_c=Delta_c, g_s=g_s, g_as=g_as, 
                kappa_s=kappa_s, kappa_as=kappa_as, Gamma_41=None, Gamma_42=None, Gamma_d=None, solver=None)
